Task 1: A* for the 8-Puzzle (Updated Puzzle)


In [1]:
from heapq import heappush, heappop

# ─── Configurable puzzle ───────────────────────────────────────────────
PUZZLE_SIZE = 3          # Change to 4 for a 15-puzzle, etc.

# Custom start and goal states (0 = blank tile)
START = (
    1, 2, 5,
    3, 4, 0,
    6, 7, 8
)

GOAL = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0
)
# ───────────────────────────────────────────────────────────────────────

N = PUZZLE_SIZE

def manhattan(state):
    """Sum of Manhattan distances of each tile from its goal position."""
    goal_pos = {val: (i // N, i % N) for i, val in enumerate(GOAL)}
    dist = 0
    for i, val in enumerate(state):
        if val != 0:
            gr, gc = goal_pos[val]
            cr, cc = i // N, i % N
            dist += abs(gr - cr) + abs(gc - cc)
    return dist

def get_neighbors(state):
    """Generate successor states by sliding a tile into the blank."""
    state = list(state)
    blank = state.index(0)
    r, c = blank // N, blank % N
    neighbors = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < N and 0 <= nc < N:
            new_state = state[:]
            swap = nr * N + nc
            new_state[blank], new_state[swap] = new_state[swap], new_state[blank]
            neighbors.append(tuple(new_state))
    return neighbors

def a_star(start, goal):
    """A* search returning the solution path and stats."""
    open_set = []
    heappush(open_set, (manhattan(start), 0, start, [start]))
    visited = {}

    nodes_expanded = 0

    while open_set:
        f, g, current, path = heappop(open_set)

        if current == goal:
            return path, nodes_expanded

        if current in visited and visited[current] <= g:
            continue
        visited[current] = g
        nodes_expanded += 1

        for neighbor in get_neighbors(current):
            new_g = g + 1
            new_h = manhattan(neighbor)
            new_f = new_g + new_h
            heappush(open_set, (new_f, new_g, neighbor, path + [neighbor]))

    return None, nodes_expanded  # No solution

def print_state(state):
    for i in range(N):
        print(state[i*N:(i+1)*N])
    print()

# ─── Run ───────────────────────────────────────────────────────────────
solution, expanded = a_star(START, GOAL)

if solution:
    print(f"✅ Solution found in {len(solution) - 1} moves ({expanded} nodes expanded)\n")
    for step, state in enumerate(solution):
        print(f"Step {step}:")
        print_state(state)
else:
    print("❌ No solution found.")

✅ Solution found in 21 moves (861 nodes expanded)

Step 0:
(1, 2, 5)
(3, 4, 0)
(6, 7, 8)

Step 1:
(1, 2, 5)
(3, 4, 8)
(6, 7, 0)

Step 2:
(1, 2, 5)
(3, 4, 8)
(6, 0, 7)

Step 3:
(1, 2, 5)
(3, 0, 8)
(6, 4, 7)

Step 4:
(1, 2, 5)
(0, 3, 8)
(6, 4, 7)

Step 5:
(1, 2, 5)
(6, 3, 8)
(0, 4, 7)

Step 6:
(1, 2, 5)
(6, 3, 8)
(4, 0, 7)

Step 7:
(1, 2, 5)
(6, 3, 8)
(4, 7, 0)

Step 8:
(1, 2, 5)
(6, 3, 0)
(4, 7, 8)

Step 9:
(1, 2, 5)
(6, 0, 3)
(4, 7, 8)

Step 10:
(1, 2, 5)
(0, 6, 3)
(4, 7, 8)

Step 11:
(0, 2, 5)
(1, 6, 3)
(4, 7, 8)

Step 12:
(2, 0, 5)
(1, 6, 3)
(4, 7, 8)

Step 13:
(2, 5, 0)
(1, 6, 3)
(4, 7, 8)

Step 14:
(2, 5, 3)
(1, 6, 0)
(4, 7, 8)

Step 15:
(2, 5, 3)
(1, 0, 6)
(4, 7, 8)

Step 16:
(2, 0, 3)
(1, 5, 6)
(4, 7, 8)

Step 17:
(0, 2, 3)
(1, 5, 6)
(4, 7, 8)

Step 18:
(1, 2, 3)
(0, 5, 6)
(4, 7, 8)

Step 19:
(1, 2, 3)
(4, 5, 6)
(0, 7, 8)

Step 20:
(1, 2, 3)
(4, 5, 6)
(7, 0, 8)

Step 21:
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)



Task 2: A* for Shortest Path in a Weighted Graph

In [2]:
import heapq
import math

# ─── Graph definition ──────────────────────────────────────────────────
# (city name): (x, y) coordinates for heuristic (straight-line distance)
COORDS = {
    "Arad":           (91, 492),
    "Zerind":         (75, 449),
    "Oradea":         (131, 571),
    "Sibiu":          (207, 457),
    "Timisoara":      (94, 410),
    "Lugoj":          (165, 379),
    "Mehadia":        (168, 339),
    "Drobeta":        (130, 285),
    "Craiova":        (253, 288),
    "Pitesti":        (325, 368),
    "Rimnicu Vilcea": (268, 430),
    "Fagaras":        (305, 478),
    "Bucharest":      (400, 327),
    "Giurgiu":        (375, 270),
    "Urziceni":       (456, 350),
    "Hirsova":        (534, 350),
    "Eforie":         (554, 300),
    "Vaslui":         (510, 433),
    "Iasi":           (473, 506),
    "Neamt":          (406, 537),
}

# Weighted edges: (city_a, city_b, distance_km)
EDGES = [
    ("Arad",           "Zerind",          75),
    ("Arad",           "Sibiu",           140),
    ("Arad",           "Timisoara",       118),
    ("Zerind",         "Oradea",          71),
    ("Oradea",         "Sibiu",           151),
    ("Timisoara",      "Lugoj",           111),
    ("Lugoj",          "Mehadia",         70),
    ("Mehadia",        "Drobeta",         75),
    ("Drobeta",        "Craiova",         120),
    ("Craiova",        "Rimnicu Vilcea",  146),
    ("Craiova",        "Pitesti",         138),
    ("Sibiu",          "Rimnicu Vilcea",  80),
    ("Sibiu",          "Fagaras",         99),
    ("Rimnicu Vilcea", "Pitesti",         97),
    ("Fagaras",        "Bucharest",       211),
    ("Pitesti",        "Bucharest",       101),
    ("Bucharest",      "Giurgiu",         90),
    ("Bucharest",      "Urziceni",        85),
    ("Urziceni",       "Hirsova",         98),
    ("Hirsova",        "Eforie",          86),
    ("Urziceni",       "Vaslui",          142),
    ("Vaslui",         "Iasi",            92),
    ("Iasi",           "Neamt",           87),
]

def build_graph(edges):
    graph = {}
    for a, b, w in edges:
        graph.setdefault(a, []).append((b, w))
        graph.setdefault(b, []).append((a, w))
    return graph

def euclidean(city_a, city_b):
    """Straight-line distance heuristic using coordinates."""
    x1, y1 = COORDS[city_a]
    x2, y2 = COORDS[city_b]
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

def a_star_graph(graph, start, goal):
    """A* search on the Romania road map."""
    h = lambda n: euclidean(n, goal)

    open_set = [(h(start), 0, start, [start])]
    visited = {}

    nodes_expanded = 0

    while open_set:
        f, g, current, path = heapq.heappop(open_set)

        if current == goal:
            return path, g, nodes_expanded

        if current in visited:
            continue
        visited[current] = g
        nodes_expanded += 1

        for neighbor, cost in graph.get(current, []):
            if neighbor not in visited:
                new_g = g + cost
                new_f = new_g + h(neighbor)
                heapq.heappush(open_set, (new_f, new_g, neighbor, path + [neighbor]))

    return None, float('inf'), nodes_expanded  # No path

# ─── Run ───────────────────────────────────────────────────────────────
GRAPH  = build_graph(EDGES)
START  = "Arad"
GOAL   = "Bucharest"

path, total_cost, expanded = a_star_graph(GRAPH, START, GOAL)

if path:
    print(f"✅ Shortest path from {START} to {GOAL}:")
    print(" → ".join(path))
    print(f"\nTotal distance : {total_cost} km")
    print(f"Nodes expanded : {expanded}")
else:
    print("❌ No path found.")

✅ Shortest path from Arad to Bucharest:
Arad → Sibiu → Rimnicu Vilcea → Pitesti → Bucharest

Total distance : 418 km
Nodes expanded : 5
